<a href="https://colab.research.google.com/github/LikithaSrinivasa25/ml-dl-algorithms/blob/main/nlp/seq2seq_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Dense, SimpleRNN

In [ ]:
batch_size = 64
epochs =12
latent_dim = 256
num_samples  = 10000

In [ ]:
data='/content/ceb seq.txt'
input_texts=[]
target_texts=[]
input_char=set()
target_char=set()
with open(data, "r", encoding = "utf-8") as f:
  lines=f.read().split("\n")
for line in lines[: min(num_samples, len(lines)-1)]:
  input_text, target_text, unwanted = line.split("\t")
  target_text = "\t" + target_text + "\n"
  input_texts.append(input_text)
  target_texts.append(target_text)
  for char in input_text:
    if char not in input_char:
      input_char.add(char)
  for char in target_text:
    if char not in target_char:
      target_char.add(char)

In [ ]:
input_char=sorted(list(input_char))
target_char=sorted(list(target_char))
num_en_token=len(input_char)
num_de_token=len(target_char)

In [ ]:
max_en_seq_len=max([len(txt) for txt in input_texts])
max_de_seq_len=max([len(txt) for txt in target_texts])

In [ ]:
print("Number of samples = ",len(input_texts))
print("Number of unique input tokens : ",num_en_token)
print("Number of unique target tokens : ",num_de_token)
print("Max sequence length for inputs : ",max_en_seq_len)
print("Max sequence length for outputs : ",max_de_seq_len)

Number of samples =  207
Number of unique input tokens :  57
Number of unique target tokens :  52
Max sequence length for inputs :  93
Max sequence length for outputs :  154


In [ ]:
input_token_index = dict(
    [(char,i) for i,char in enumerate(input_char)])
target_token_index=dict(
    [(char,i) for i,char in enumerate(target_char)])

In [ ]:
encoder_input_data=np.zeros(
    (len(input_texts),max_de_seq_len,num_en_token),dtype='float32')
decoder_input_data=np.zeros(
  (len(input_texts),max_de_seq_len,num_de_token),dtype='float32')
decoder_output_data=np.zeros(
    (len(input_texts),max_de_seq_len,num_de_token),dtype='float32')

In [ ]:
for i,(input_text,target_text) in enumerate(zip(input_texts,target_texts)):
  for t,char in enumerate(input_text):
    encoder_input_data[i,t,input_token_index[char]]==1.
  encoder_input_data[i,t+1:,input_token_index[' ']]==1.
  for t,char in enumerate(target_text):
    decoder_input_data[i,t,target_token_index[char]]==1.
    if t>0:
      decoder_output_data[i,t-1:,target_token_index[char]]==1.
  decoder_input_data[i,t+1:,target_token_index[' ']]==1.
  decoder_output_data[i,t:,target_token_index[' ']]==1.

In [ ]:
encoder_inputs= Input(shape=(None,num_en_token))
encoder=LSTM(latent_dim,return_state=True)
encoder_outputs,state_h,state_c=encoder(encoder_inputs)
encoder_states=[state_h,state_c]


In [ ]:
decoder_input=Input(shape=(None,num_de_token))
decoder_lstm=LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_input,initial_state=encoder_states)

decoder_dense = Dense(num_de_token, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
model=Model([encoder_inputs,decoder_input],decoder_outputs)

In [ ]:
model.compile(optimizer='rmsprop',loss='categorical_crossentropy')
model.fit([encoder_input_data,decoder_input_data],decoder_output_data,batch_size=batch_size
          ,verbose=0,epochs=epochs,validation_split=0.2)

In [ ]:
input_text,target_text

('Martin Luther King Jr. fought for civil rights for African-Americans by teaching nonviolence.',
 '\tPinaagi sa pagtudlo sa pagka dili madinagmalon nakigbisog si Martin Luther King Jr. alang sa katungod nga sibil sa mga Amerkanong may dugong Afrikanhon.\n')

In [ ]:
encoder_inputs = model.input[0]
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output
encoder_states = [state_h_enc, state_c_enc]
encoder_model = Model(encoder_inputs, encoder_states)

In [ ]:
decoder_inputs = model.input[1]
decoder_state_input_h = Input(shape=(latent_dim,),name='input_3') # state_h
decoder_state_input_c = Input(shape=(latent_dim,),name='input_4') # state_c
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]

In [ ]:
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs=decoder_dense(decoder_outputs)

In [ ]:
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)

In [ ]:
#Reverse lookup tokens to decode encoded sequence
reverse_input_char_index = dict(
    (i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict(
    (i, char) for char, i in target_token_index.items())

In [ ]:
def decode_seq(input_seq):
  states_value=encoder_model.predict(input_seq)
  target_seq=np.zeros((1,1,num_de_token))
  target_seq[0,0,target_token_index['\t']]=1.0
  stop_condition=False
  decoded_sentence=''
  while not stop_condition:
    output_tokens,h,c=decoder_model.predict([target_seq]+states_value)
    sampled_token_index = np.argmax(output_tokens[0, -1, :])
    sampled_char = reverse_target_char_index[sampled_token_index]
    decoded_sentence += sampled_char
    if (sampled_char == '\n' or len(decoded_sentence) > max_de_seq_len):
      stop_condition = True
    target_seq = np.zeros((1, 1, num_de_token))
    target_seq[0, 0, sampled_token_index] = 1.0
    states_value = [h, c]
  return decoded_sentence

In [ ]:
for seq_index in range(3):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_seq(input_seq)
    print('-')
    print('Input sentence:', input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)

1/1 [==============================] - 0s 24ms/step
-
Input sentence: Hello!
Decoded sentence: JJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJ
1/1 [==============================] - 0s 50ms/step
-
Input sentence: Oh no!
Decoded sentence: JJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJ
1/1 [==============================] - 0s 27ms/step
-
Input sentence: No way!
Decoded sentence: JJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJJ
